In [ ]:
# %pip install --upgrade pip
# %pip install transformers==4.40.1 peft==0.4.0
# %pip install sentencepiece
# %pip install accelerate
# %pip install torch
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
# %pip install peft
# %pip install datasets
# %pip install bitsandbytes
# %pip install huggingface_hub

# import os
# from huggingface_hub import login
# # from .autonotebook import tqdm as notebook_tqdm

# login(token=os.getenv("HF_TOKEN"))


In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    'meta-llama/Llama-2-7b-chat-hf',
    trust_remote_code=True,
    device_map="cuda:0",
    torch_dtype=torch.float16,   # optional if you have enough VRAM
)
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-2-7b-chat-hf')

model = PeftModel.from_pretrained(base_model, 'FinGPT/fingpt-forecaster_dow30_llama2-7b_lora')
model = model.eval()


c:\_Coding\LLM-Predict-Stock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\_Coding\LLM-Predict-Stock\.venv\Lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:32<00:00, 16.26s/it]


In [ ]:
# Make prompts
prompt = [
# '''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
# Input: 川普關稅政策帶來市場亂流，此外，密西根大學調查顯示，2月消費者預期未來1年通膨為4.3%，創2023年11月以來新高
# Answer: ''',
'''
[Cryptocurrency Introduction]: Bitcoin (BTC) is a cryptocurrency launched in 2010. Users are able to generate BTC through the process of mining. Bitcoin has a current supply of 19,599,987. The last known price of Bitcoin is 42,945.87509791 USD and is up 0.58 over the last 24 hours. It is currently trading on 10709 active market(s) with $21,150,003,121.31 traded over the last 24 hours. More information can be found at https://bitcoin.org/.. It has a market capilization of 845807222784.

From 2023-11-12 to 2023-11-19, BTC-USD's stock price increased from 37054.52 to 37386.55. Company news during this period are listed below:

[Headline]: Bitcoin, BlackRock, And The End Of Decentralization
[Summary]: BlackRock's entry into cryptocurrency marks the end of the idea of a decentralized world currency. Learn more about Bitcoin's rally in 2023.

[Headline]: Bullish Litecoin, But Be Careful Chasing The Grayscale Litecoin Trust
[Summary]: Grayscale Litecoin Investment Trust has seen a surge in demand bring the trust's price from a NAV discount to a NAV premium in recent sessions. Read why LTCN is a Hold.

[Headline]: Riot Platforms: Poised To Ride The Bitcoin Wave To New Heights
[Summary]: Riot Platforms, one of the largest Bitcoin miners, is well-positioned to ride the wave higher as Bitcoin continues to be bid higher. Read my analysis of RIOT.
'''
]

# Set device
tokenizer.pad_token = tokenizer.eos_token


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)
tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512).to(device)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
print(res_sentences)
out_text = [o.split("Answer: ")[1] for o in res_sentences]


# Show results
for sentiment in out_text:
    print(sentiment)

cuda
["<s> [Cryptocurrency Introduction]: Bitcoin (BTC) is a cryptocurrency launched in 2010. Users are able to generate BTC through the process of mining. Bitcoin has a current supply of 19,599,987. The last known price of Bitcoin is 42,945.87509791 USD and is up 0.58 over the last 24 hours. It is currently trading on 10709 active market(s) with $21,150,003,121.31 traded over the last 24 hours. More information can be found at https://bitcoin.org/.. It has a market capilization of 845807222784.\n\nFrom 2023-11-12 to 2023-11-19, BTC-USD's stock price increased from 37054.52 to 37386.55. Company news during this period are listed below:\n\n[Headline]: Bitcoin, BlackRock, And The End Of Decentralization\n[Summary]: BlackRock's entry into cryptocurrency marks the end of the idea of a decentralized world currency. Learn more about Bitcoin's rally in 2023.\n\n[Headline]: Bullish Litecoin, But Be Careful Chasing The Grayscale Litecoin Trust\n[Summary]: Grayscale Litecoin Investment Trust has

IndexError: list index out of range

In [1]:
import os
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast, BitsAndBytesConfig
from peft import PeftModel  # 0.5.0
import torch

# Load Models
base_model = "meta-llama/Meta-Llama-3-8B" 
peft_model = "FinGPT/fingpt-mt_llama3-8b_lora"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

offload_dir = 'C:\_Coding\LLM-Predict-Stock\run_finetune\offload\dir'

model = LlamaForCausalLM.from_pretrained(
    base_model,
    trust_remote_code=True,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    offload_folder=offload_dir  # Add offload_dir here for the base model
)

# Load PEFT adapter
model = PeftModel.from_pretrained(
    model,
    peft_model,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    offload_folder=offload_dir  # Add offload_dir here for the PEFT model
)

model = model.eval()

c:\_Coding\LLM-Predict-Stock\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\_Coding\LLM-Predict-Stock\.venv\Lib\site-packages\huggingface_hub\file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'PreTrainedTokenizerFast'. 
The class this function is called from is 'LlamaTokenizerFast'.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
The argument `trust_remote

OSError: [WinError 123] 檔案名稱、目錄名稱或磁碟區標籤語法錯誤。: 'C:\\_Coding\\LLM-Predict-Stock\run_finetune'

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast
from peft import PeftModel  # 0.5.0
import torch

# Load Models
base_model = "meta-llama/Meta-Llama-3-8B" 
peft_model = "FinGPT/fingpt-mt_llama3-8b_lora"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cuda:0")
model = PeftModel.from_pretrained(model, peft_model)
model = model.eval()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512).to(device)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]

# Make prompts
prompt = [
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is aggressively pursuing its growth strategy by increasingly focusing on technologically more demanding HDI printed circuit boards PCBs .
Answer: ''',
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
Answer: '''
]



# Show results
for sentiment in out_text:
    print(sentiment)


In [3]:
%%time

# Set device
tokenizer.pad_token = tokenizer.eos_token


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)
model.half()  # 將模型轉換為半精度
tokens = tokens.half()

tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512).to(device)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]





# Show results
for sentiment in out_text:
    print(sentiment)

cuda


NameError: name 'model' is not defined

In [4]:
# Make prompts
prompt = [
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: 川普關稅政策帶來市場亂流，此外，密西根大學調查顯示，2月消費者預期未來1年通膨為4.3%，創2023年11月以來新高
Answer: ''',
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: Apple Seeks to End Savings & Credit Card Partnership With Goldman Sachs — What Will Happen to Your Accounts?
Answer: '''
]

# Set device
tokenizer.pad_token = tokenizer.eos_token


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512).to(device)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]





# Show results
for sentiment in out_text:
    print(sentiment)

NameError: name 'model' is not defined